# Customer Churn Data Cleaning
Dataset: IBM Telco Customer Churn (`telco_churn.csv`)

This notebook walks through cleaning the raw CSV step by step. Run each cell in order,
check the output, then move to the next. At the end we save a clean CSV ready for
SQL / Postgres / Power BI.

## 1. Load the raw data and take a first look

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('telco_churn.csv')
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Check data types and non-null counts — this is where you spot problems
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 2. Check for missing values and duplicates
`TotalCharges` looks numeric but pandas read it as text (`object`) — that's usually
a sign of blank strings or stray characters hiding in the column.

In [ ]:
print("Nulls per column:")
print(df.isnull().sum())

print("\nDuplicate customerIDs:", df['customerID'].duplicated().sum())

# TotalCharges is read as object (text) — find out why
print("\nNon-numeric TotalCharges values:")
print(df[pd.to_numeric(df['TotalCharges'], errors='coerce').isna()]['TotalCharges'].unique())

Nulls per column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Duplicate customerIDs: 0

Non-numeric TotalCharges values:
<StringArray>
[' ']
Length: 1, dtype: str


## 3. Fix `TotalCharges`
The blanks are all brand-new customers (`tenure == 0`). We convert to numeric and
fill those blanks with their `MonthlyCharges` (their first bill).

In [ ]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan).astype(float)

# confirm the blanks line up with tenure == 0
print(df[df['TotalCharges'].isna()][['customerID', 'tenure', 'MonthlyCharges']])

df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])
print("\nRemaining nulls in TotalCharges:", df['TotalCharges'].isna().sum())

      customerID  tenure  MonthlyCharges
488   4472-LVYGI       0           52.55
753   3115-CZMZD       0           20.25
936   5709-LVOEQ       0           80.85
1082  4367-NUYAO       0           25.75
1340  1371-DWPAZ       0           56.05
3331  7644-OMVMY       0           19.85
3826  3213-VVOLG       0           25.35
4380  2520-SGTTA       0           20.00
5218  2923-ARZLG       0           19.70
6670  4075-WKNIU       0           73.35
6754  2775-SEFEE       0           61.90

Remaining nulls in TotalCharges: 0


## 4. Standardize flag columns
`SeniorCitizen` is 0/1 while every other Yes/No column is text — make it consistent.

In [ ]:
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df['SeniorCitizen'].value_counts()

SeniorCitizen
No     5901
Yes    1142
Name: count, dtype: int64

## 5. Engineer analysis-friendly columns
Bucket `tenure` and `MonthlyCharges` into readable ranges, and add a numeric
`ChurnFlag` (0/1) for easy averaging in SQL and Power BI.

In [ ]:
def tenure_bucket(t):
    if t <= 12: return '0-1 yr'
    elif t <= 24: return '1-2 yr'
    elif t <= 48: return '2-4 yr'
    elif t <= 60: return '4-5 yr'
    else: return '5+ yr'

def charge_bucket(c):
    if c < 35: return 'Low (<$35)'
    elif c < 70: return 'Medium ($35-70)'
    elif c < 100: return 'High ($70-100)'
    else: return 'Very High ($100+)'

df['TenureBucket'] = df['tenure'].apply(tenure_bucket)
df['ChargeBucket'] = df['MonthlyCharges'].apply(charge_bucket)
df['ChurnFlag'] = (df['Churn'] == 'Yes').astype(int)

df[['tenure', 'TenureBucket', 'MonthlyCharges', 'ChargeBucket', 'Churn', 'ChurnFlag']].head()

,tenure,TenureBucket,MonthlyCharges,ChargeBucket,Churn,ChurnFlag
0,1,0-1 yr,29.85,Low (<$35),No,0
1,34,2-4 yr,56.95,Medium ($35-70),No,0
2,2,0-1 yr,53.85,Medium ($35-70),Yes,1
3,45,2-4 yr,42.30,Medium ($35-70),No,0
4,2,0-1 yr,70.70,High ($70-100),Yes,1


## 6. Sanity-check the cleaned data before saving

In [ ]:
print(df.shape)
print("\nRemaining nulls:", df.isnull().sum().sum())
print("\nChurn distribution:")
print(df['Churn'].value_counts(normalize=True).round(3))
df.describe(include='all').T

(7043, 24)

Remaining nulls: 0

Churn distribution:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043,2,No,5901,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Save the clean file

In [ ]:
df.to_csv('telco_churn_clean.csv', index=False)
print("Saved telco_churn_clean.csv with", len(df), "rows and", len(df.columns), "columns")

Saved telco_churn_clean.csv with 7043 rows and 24 columns
